In [ ]:
import h5py
import shutil
import os

def trim_obs_var_inplace(h5ad_path: str, keep_obs: list[str], keep_var: list[str]):
    
    with h5py.File(h5ad_path, "r+") as f:
        for group_name, keep in [("obs", keep_obs), ("var", keep_var)]:
            grp = f[group_name]
            existing = [k for k in grp.keys() if k != "_index"]
            to_delete = [k for k in existing if k not in keep]

            print(f"\n[{group_name}] Keeping {keep}, deleting {to_delete}")
            for col in to_delete:
                del grp[col]

def copy_and_trim_h5ad(
    input_path: str,
    output_path: str,
    keep_obs: list[str],
    keep_var: list[str],
    overwrite: bool = False
):

    if os.path.exists(output_path):
        if overwrite:
            os.remove(output_path)
        else:
            raise FileExistsError(f"{output_path} already exists. Use overwrite=True to replace.")

    print(f"Copying {input_path} → {output_path} ...")
    shutil.copy2(input_path, output_path)

    print(f"Trimming metadata in {output_path} ...")
    trim_obs_var_inplace(output_path, keep_obs, keep_var)

In [ ]:
keep_obs = ['cell_id', 'donor_id', 'site_id', 'Sample', 'NucleosomeRatio', 'PromoterRatio', 'ReadsInPeaks', 
            'ReadsInPromoter', 'ReadsInTSS', 'TSSEnrichment', 'nFrags']
keep_var = ['seqnames', 'start','end','GC','score', 'width', 'peakType']

In [ ]:
copy_and_trim_h5ad(
    input_path="../data/multiome/atac_filtered_peaks.h5ad",
    output_path="../data/multiome/atac_filtered_peaks.h5ad",
    keep_obs=keep_obs,
    keep_var=keep_var,
)

In [ ]:
def inspect_anndata_h5ad(h5ad_path: str):
    print(f"Size: {os.path.getsize(h5ad_path) / 1e6:.2f} MB")

    with h5py.File(h5ad_path, "r") as f:
        for key in f.keys():
            item = f[key]
            if isinstance(item, h5py.Dataset):
                print(f"  {key}/ → dataset, shape={item.shape}, dtype={item.dtype}")
            elif isinstance(item, h5py.Group):
                print(f"  {key}/ → group")
            else:
                print(f"  {key}/ → unknown type")

        # Inspect X
        if "X" in f:
            x = f["X"]
            print(f"  Type: {'sparse' if isinstance(x, h5py.Group) else 'dense'}")
            if hasattr(x, "shape"):
                print(f"  Shape: {x.shape}")
            if isinstance(x, h5py.Group):
                print(f"  Sparse format keys: {list(x.keys())}")

        # Inspect obs
        if "obs" in f:
            for key in f["obs"].keys():
                obj = f["obs"][key]
                if isinstance(obj, h5py.Dataset):
                    print(f"  - {key}: shape={obj.shape}, dtype={obj.dtype}")
                elif isinstance(obj, h5py.Group):
                    print(f"  - {key}/: group (e.g., Categorical)")
                else:
                    print(f"  - {key}: unknown type")

        # Inspect var
        if "var" in f:
            for key in f["var"].keys():
                obj = f["var"][key]
                if isinstance(obj, h5py.Dataset):
                    print(f"  - {key}: shape={obj.shape}, dtype={obj.dtype}")
                elif isinstance(obj, h5py.Group):
                    print(f"  - {key}/: group (e.g., Categorical)")
                else:
                    print(f"  - {key}: unknown type")

In [ ]:
inspect_anndata_h5ad("../data/multiome/atac_filtered_peaks.h5ad")